# Linear Regression

Linear regression is used when there's a clear linear relationship between a dependent (output) target variable and an independent (input) variable. The result is a line of best fit that tries to get as close to as many points as possible.

---

## The Model

$$ f_{w,b}(x) = w \cdot x + b $$

Where each variable represents:

- $f_{w,b}(x)$: The predicted y value according to the given inputs
- $b$: The y-intercept (the bias)
- $w$: The slope of the line of best fit (the weight)
- $x$: The input value

---

## The Loss / Cost Function

To find the line of best fit we need to calculate the loss, which measures how far the line is from the actual results. The loss function used is Mean Squared Error (MSE), which measures the error by squaring the difference between the predicted and actual value of y. By summing all squared errors and dividing by $2m$ we get the average squared error of the line, telling us how wrong it is on average. The $\frac{1}{2}$ is a convenience factor, it cancels out cleanly when we take the derivative later. Our goal is to minimize this cost.

$$J(w,b) = \frac{1}{2m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)})^2$$

Where:

- $J(w,b)$: The cost
- $m$: The number of training examples
- $i$: Denotes which example we are on
- $x^{(i)}$: The $i$ th input value
- $y^{(i)}$: The $i$ th actual output value

---

## Gradient Descent

To minimize the cost we use gradient descent. We compute the partial derivatives of $J(w,b)$ with respect to $w$ and $b$, these tell us the slope of the cost function at the current values. Since the gradient points uphill, we subtract it from our parameters to move downhill toward the minimum. $\alpha$ is the learning rate, controlling how large each step is.

The update rules are:

$$w := w - \alpha \frac{\partial J(w,b)}{\partial w}$$

$$b := b - \alpha \frac{\partial J(w,b)}{\partial b}$$

Where the partial derivatives expand to:

$$\frac{\partial J(w,b)}{\partial w} = \frac{1}{m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)}) x^{(i)}$$

$$\frac{\partial J(w,b)}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)})$$

Notice the derivative for $w$ has an extra $x^{(i)}$ term compared to $b$, which comes from the chain rule, since $w$ is multiplied by $x$ in the model.

---

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression as lr
from sklearn.metrics import mean_squared_error, r2_score
import pickle

def standardize_x(x_new):
    return (x_new - np.mean(x_train)) / np.std(x_train)

def standardize_y(y_new):
    return (y_new - np.mean(y_train)) / np.std(y_train)

class LinearRegression:
    
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate
        self.w = np.random.randn() * .01
        self.b = 0
    
    def predict(self, x_train):
        return x_train * self.w + self.b

    def loss(self, x_train, y_train):
        m = len(x_train)
        loss = (1/(2*m))* (np.sum((self.predict(x_train) - y_train)**2))
        return loss
    
    def gradient_descent(self, x_train, y_train):
        m = len(x_train)
        weight_partial = (1/m) * np.sum((self.predict(x_train)-y_train) * x_train)
        bias_partial = (1/m) * np.sum(self.predict(x_train)-y_train)
        
        self.w -= self.learning_rate * weight_partial
        self.b -= self.learning_rate * bias_partial
    
    def fit(self, x_train, y_train, epochs):
        self.x = x_train
        self.y = y_train
        
        losses = []
        
        for i in range(epochs):
            loss = self.loss(x_train, y_train)
            self.gradient_descent(x_train, y_train)
            losses.append(loss)
        
        fig = px.line(y=losses, title="Loss vs Iteration", template="plotly_dark")
        fig.update_layout(
            title_font_color="#41BEE9",
            xaxis=dict(color="#41BEE9", title="Iterations"),
            yaxis=dict(color="#41BEE9", title="Loss")
        )

        fig.show()
            
data = pd.read_csv('../data/Salary_dataset.csv')
test_datasets = {
    'Test Set 1': pd.DataFrame({
        'YearsExperience': [1.0, 1.8, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.0, 10.0, 11.0, 12.0],
        'Salary':          [41000, 38500, 55000, 52000, 71000, 69000, 79000, 102000, 95000, 118000, 108000, 125000, 142000]
    }),
    'Test Set 2': pd.DataFrame({ #contains outliers
        'YearsExperience': [1.0, 1.8, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.0, 10.0, 11.0, 12.0],
        'Salary':          [41000, 38500, 55000, 52000, 71000, 150000, 79000, 102000, 28000, 118000, 108000, 195000, 142000]
    }),
    'Test Set 3': pd.DataFrame({
        'YearsExperience': [1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.5, 10.5, 11.5, 12.5, 13.0],
        'Salary':          [40000, 45000, 53000, 61000, 72000, 85000, 93000, 99000, 115000, 119000, 131000, 139000, 145000]
    }),
    'Test Set 4': pd.DataFrame({ #This one is non-linear
    'YearsExperience': [1.0, 1.8, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.0, 10.0, 11.0, 12.0],
    'Salary':          [90000, 85000, 88000, 40000, 42000, 38000, 92000, 95000, 41000, 88000, 44000, 91000, 39000]
})
}

x_train = data['YearsExperience']
y_train = data['Salary']

fig = px.scatter(data, x='YearsExperience', y='Salary', title="Salary vs Years Experience")
fig.show() #Plot out the points of the data

x_train_scaled= standardize_x(x_train)
y_train_scaled = standardize_y(y_train)


model = LinearRegression(.01)
model.fit(x_train_scaled, y_train_scaled, 500)

y_pred_scaled = model.predict(x_train_scaled)
y_pred = (y_pred_scaled * np.std(y_train)) + np.mean(y_train)
fig.add_scatter(x=data['YearsExperience'], y=y_pred, mode='lines', name='fit')
fig.show()
        

Iteration: 0, Loss: 0.49465351706811717
Iteration: 100, Loss: 0.08491171924095203
Iteration: 200, Loss: 0.03001464639235169
Iteration: 300, Loss: 0.022659554421442293
Iteration: 400, Loss: 0.021674121590629477


In [2]:
#training skikitlearn lr on same data
sk_model = lr()
sk_model.fit(x_train_scaled.values.reshape(-1,1), y_train_scaled)

#outputting line of best fit on differnt test datas
for dataset_name, df in test_datasets.items():
    x_test = df['YearsExperience']
    y_test = df['Salary']

    x_test_scaled = standardize_x(x_test)

    y_pred_scaled = model.predict(x_test_scaled)
    y_pred = (y_pred_scaled * np.std(y_train)) + np.mean(y_train)

    sk_pred_test = (sk_model.predict(x_test_scaled.values.reshape(-1, 1)) * np.std(y_train)) + np.mean(y_train)

    fig = px.scatter(x=x_test, y=y_test, title=f'{dataset_name} — Model Predictions',
                     template="plotly_dark", labels={'x': 'YearsExperience', 'y': 'Salary'})
    fig.add_scatter(x=x_test, y=y_pred, mode='lines', name='fit')
    fig.update_layout(title_font_color="#41BEE9",
                      xaxis=dict(color="#41BEE9"),
                      yaxis=dict(color="#41BEE9"))
    fig.show()

    ss_res = np.sum((y_test - y_pred)**2)
    ss_tot = np.sum((y_test - np.mean(y_test))**2)
    r2 = 1 - (ss_res / ss_tot)
    print(f'My R²:      {r2:.4f}')
    print(f'Sklearn R²: {r2_score(y_test, sk_pred_test):.4f}\n')

My R²:      0.9539
Sklearn R²: 0.9533



My R²:      0.4724
Sklearn R²: 0.4729



My R²:      0.9957
Sklearn R²: 0.9950



My R²:      -2.8395
Sklearn R²: -2.8698



$R^2$ represents how accurately the line of best fit matches the data. 
A value of 1 is a perfect fit, 0 means the model is no better than 
guessing the mean, and a negative value means it performs worse than 
just guessing the mean.

My $R^2$ values almost exactly match sklearn's across all test sets, 
demonstrating that the from-scratch implementation is correct.

Test Sets 1 and 3 have high $R^2$ values as the data follows a clean 
linear trend consistent with what the model was trained on.

Test Set 2 has a lower $R^2$ due to outliers pulling points far from 
the line. Since the training data contained no outliers, the model 
never learned to account for them.

Test Set 4 has a negative $R^2$, meaning the line of best fit is 
actively worse than just predicting the mean salary for everyone. 
The relationship in the data is non-linear and would require a 
different model entirely, such as polynomial regression.